# S07 — Regularization, Initialization, Normalization

**Week 4 · Wed Sep 16, 2026 · Module 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s07_regularization_initialization_normalization.ipynb)

Every cell below is a worked example from the [S07 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s07/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s07.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s07.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
try:
    import torch  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch  # noqa: F401
print("environment ready")

## Overfitting, weight decay, and early stopping


*Expected output starts with:* ` weight_decay  train MSE  final val   best val  best epoch`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# 30 noisy training points from y = sin(2x); a big MLP can memorize them.
x_train = torch.rand(30, 1) * 6 - 3
y_train = torch.sin(2 * x_train) + 0.3 * torch.randn(30, 1)
x_val = torch.linspace(-3, 3, 200).unsqueeze(1)
y_val = torch.sin(2 * x_val)            # noise-free target for evaluation

def train(weight_decay):
    torch.manual_seed(1)
    model = nn.Sequential(
        nn.Linear(1, 128), nn.Tanh(),
        nn.Linear(128, 128), nn.Tanh(),
        nn.Linear(128, 1),
    )
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    best_val, best_epoch = float("inf"), 0
    for epoch in range(1, 4001):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(x_train), y_train)
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            val = loss_fn(model(x_val), y_val).item()
        if val < best_val:
            best_val, best_epoch = val, epoch
    return loss.item(), val, best_val, best_epoch

print(f"{'weight_decay':>13s}{'train MSE':>11s}{'final val':>11s}{'best val':>11s}{'best epoch':>12s}")
for wd in (0.0, 1e-1, 1.0, 3.0):
    tr, val, bv, be = train(wd)
    print(f"{wd:13.0e}{tr:11.4f}{val:11.4f}{bv:11.4f}{be:12d}")

## Dropout


*Expected output starts with:* `train mode: fraction kept = 0.4954`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

drop = nn.Dropout(p=0.5)
x = torch.ones(10000)                 # activations, all equal to 1.0

drop.train()                          # training mode: random zeroing + rescale
y_train = drop(x)
kept = (y_train != 0).float().mean().item()
print(f"train mode: fraction kept = {kept:.4f}")
print(f"train mode: surviving values are   {y_train[y_train != 0][0].item():.1f}  (1 / (1-p) = 2.0)")
print(f"train mode: mean of output = {y_train.mean().item():.4f}  (expectation preserved)")

drop.eval()                           # eval mode: identity
y_eval = drop(x)
print(f"eval  mode: mean of output = {y_eval.mean().item():.4f}, zeros = {(y_eval == 0).sum().item()}")

## Initialization: why the scale of random weights matters


*Expected output starts with:* `activation std, layer:           1           5          10          20          40`


In [ ]:
import math
import torch

torch.manual_seed(0)

depth, width, batch = 40, 256, 512
x0 = torch.randn(batch, width)          # inputs with std ~ 1

def forward_stats(weight_std, label):
    """Push x0 through `depth` ReLU layers; report activation std along the way."""
    torch.manual_seed(1)                # same weights-up-to-scale for each run
    x = x0
    stds = {}
    with torch.no_grad():
        for layer in range(1, depth + 1):
            W = torch.randn(width, width) * weight_std
            x = torch.relu(x @ W.T)
            if layer in (1, 5, 10, 20, 40):
                stds[layer] = x.std().item()
    row = "".join(f"{stds[l]:>12.2e}" for l in (1, 5, 10, 20, 40))
    print(f"{label:>22s}{row}")

print(f"{'activation std, layer:':>22s}" + "".join(f"{l:>12d}" for l in (1, 5, 10, 20, 40)))
forward_stats(0.01,                      "N(0, 0.01^2)")
forward_stats(math.sqrt(1.0 / width),    "Xavier  sqrt(1/n)")
forward_stats(math.sqrt(2.0 / width),    "Kaiming sqrt(2/n)")

## Normalization: BatchNorm and LayerNorm


*Expected output starts with:* `BatchNorm: per-FEATURE mean/std across the batch (dim=0)`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# A batch of 8 samples, 4 features, deliberately different scales per feature.
x = torch.randn(8, 4) * torch.tensor([0.1, 1.0, 10.0, 100.0]) + torch.tensor([0.0, 1.0, -5.0, 50.0])

bn = nn.BatchNorm1d(4, affine=False)   # affine=False: just the normalization
ln = nn.LayerNorm(4, elementwise_affine=False)

bn.train()
y_bn = bn(x)
y_ln = ln(x)

def stats(t, dim):
    return t.mean(dim=dim), t.std(dim=dim, unbiased=False)

m, s = stats(y_bn, 0)
print("BatchNorm: per-FEATURE mean/std across the batch (dim=0)")
print("  mean:", " ".join(f"{v:8.4f}" for v in m))
print("  std :", " ".join(f"{v:8.4f}" for v in s))
m, s = stats(y_bn, 1)
print("BatchNorm: per-SAMPLE mean across features (dim=1) — NOT normalized")
print("  mean:", " ".join(f"{v:8.4f}" for v in m[:4]), "...")

m, s = stats(y_ln, 1)
print("LayerNorm: per-SAMPLE mean/std across features (dim=1)")
print("  mean:", " ".join(f"{v:8.4f}" for v in m[:4]), "...")
print("  std :", " ".join(f"{v:8.4f}" for v in s[:4]), "...")

# BatchNorm's weakness: the output for sample 0 depends on WHO ELSE is in the batch.
bn2 = nn.BatchNorm1d(4, affine=False)
bn2.train()
pair = bn2(x[[0, 7]])               # same sample 0, different batchmates
print("\nsample 0 normalized in a batch of 8 :", " ".join(f"{v:7.3f}" for v in y_bn[0]))
print("sample 0 normalized in a batch of 2 :", " ".join(f"{v:7.3f}" for v in pair[0]))
print("sample 0 under LayerNorm (batch-free):", " ".join(f"{v:7.3f}" for v in ln(x[:1])[0]))

# And with a batch of one, training-mode BatchNorm cannot even run:
try:
    bn2(x[:1])
except ValueError as e:
    print(f"\nBatchNorm1d on a single sample in train mode -> ValueError: {e}")

## The wider normalization zoo


*Expected output starts with:* `LayerNorm per-sample mean:  0.0000  0.0000  0.0000 -0.0000`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# One batch: 4 samples, 6 features, wildly different scales and offsets.
x = torch.randn(4, 6) * 3.0 + 2.0

ln = nn.LayerNorm(6, elementwise_affine=False)
rms = nn.RMSNorm(6, elementwise_affine=False)
gn = nn.GroupNorm(num_groups=2, num_channels=6, affine=False)

y_ln, y_rms, y_gn = ln(x), rms(x), gn(x)

def row_stats(t):
    mean = t.mean(dim=-1)
    rms_val = t.pow(2).mean(dim=-1).sqrt()
    return mean, rms_val

for name, y in (("LayerNorm", y_ln), ("RMSNorm", y_rms)):
    m, r = row_stats(y)
    print(f"{name:9s} per-sample mean: " + " ".join(f"{v:7.4f}" for v in m))
    print(f"{name:9s} per-sample rms : " + " ".join(f"{v:7.4f}" for v in r))

# GroupNorm with 2 groups of 3 channels: zero mean WITHIN each group, per sample.
grp = y_gn.view(4, 2, 3)
print("GroupNorm per-(sample,group) means:")
for i in range(4):
    print("  sample", i, " ".join(f"{v:7.4f}" for v in grp[i].mean(dim=-1)))

# GroupNorm and RMSNorm are batch-size independent: batch of 1 is fine in train mode.
print(f"\nGroupNorm on a single sample, train mode: shape {tuple(gn(x[:1]).shape)} (no error)")
same = torch.allclose(gn(x[:1]), y_gn[:1])
print(f"same output as in the batch of 4: {same}")

## When more parameters help: double descent


*Expected output starts with:* `n_train = 40  (interpolation threshold at n_features ~ 40)`


In [ ]:
import numpy as np

np.random.seed(0)

# Random-features regression around the interpolation threshold.
# Teacher: y = x @ w_true + noise, in d = 8 dimensions, n_train = 40 points.
d, n_train, n_test, n_trials = 8, 40, 2000, 40
X = np.random.randn(n_train, d)
X_test = np.random.randn(n_test, d)
w_true = np.random.randn(d)
y = X @ w_true + 0.5 * np.random.randn(n_train)
y_test = X_test @ w_true                      # noise-free targets for test MSE

def fit_random_features(n_feat, rng):
    V = rng.standard_normal((d, n_feat)) / np.sqrt(d)
    F = np.maximum(X @ V, 0.0)                # random ReLU features
    F_test = np.maximum(X_test @ V, 0.0)
    # lstsq returns the least-squares fit when overdetermined and the
    # MINIMUM-NORM interpolating solution when underdetermined.
    coef, *_ = np.linalg.lstsq(F, y, rcond=None)
    train_mse = np.mean((F @ coef - y) ** 2)
    test_mse = np.mean((F_test @ coef - y_test) ** 2)
    return train_mse, test_mse

print(f"n_train = {n_train}  (interpolation threshold at n_features ~ {n_train})")
print(f"{'n_features':>11s}{'train MSE':>12s}{'test MSE':>12s}")
for n_feat in (5, 10, 20, 30, 38, 40, 42, 50, 80, 160, 640):
    rng = np.random.default_rng(1)
    tr, te = np.mean([fit_random_features(n_feat, rng) for _ in range(n_trials)], axis=0)
    print(f"{n_feat:>11d}{tr:>12.4f}{te:>12.4f}")

## Try it yourself

1. In the weight-decay experiment, add dropout (`nn.Dropout(0.2)` after each hidden layer) with `weight_decay=0` and compare its final and best validation MSE against the `wd=1.0` row. Remember `model.train()` / `model.eval()` — the experiment's loop already switches modes.
2. Extend the initialization experiment to `tanh` activations. Show that Xavier now holds the standard deviation roughly steady and Kaiming's extra factor of `sqrt(2)` makes activations saturate (std pinned near the tanh output range).
3. Repeat the initialization experiment but print the *gradient* std at layer 1 by running a backward pass from the mean output. Confirm that the collapse you saw forward also appears backward.
4. Replace `BatchNorm1d` with `LayerNorm` inside a small MLP trained on the ring dataset from the [optimizers]({{ '/readings/ch1/optimizers/' | relative_url }}) section, and verify the LayerNorm model gives bit-identical outputs for batch sizes 1 and 256 at eval time while the BatchNorm model in train mode cannot even process batch size 1.
5. In the double-descent experiment, replace `lstsq` with ridge regression (solve `(F^T F + lam * I) w = F^T y` for a small `lam` such as `1e-3`) and rerun the sweep. What happens to the spike at 40 features, and what does that tell you about the relationship between explicit regularization and the interpolation threshold?


---

Full discussion of everything above: [S07 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s07/).
